In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re
from urllib.parse import urlparse
import numpy as np

In [2]:
df = pd.read_csv("DataFolder/train.csv")


**Data Augmentation**

We split each original row into 5 rows, where the the new 4 rows are made from the 2 Positive and Negative Examples.

In [3]:
rows = []

for _, row in df.iterrows():
    base = {
        "original_row_id": row["row_id"],
        "rule": row["rule"],
        "subreddit": row["subreddit"],
    }

    rows.append({        
        **base,
        "comment": row["body"],
        "rule_violation": row["rule_violation"],
        "source_type": "body"
    })

    for i, col in enumerate(["positive_example_1", "positive_example_2"], start=1):
        if pd.notna(row[col]):
            rows.append({
                **base,
                "comment": row[col],
                "rule_violation": 1,
                "source_type": f"positive_example_{i}"
            })

    for i, col in enumerate(["negative_example_1", "negative_example_2"], start=1):
        if pd.notna(row[col]):
            rows.append({
                **base,
                "comment": row[col],
                "rule_violation": 0,
                "source_type": f"negative_example_{i}"
            })


In [4]:
expanded_df = pd.DataFrame(rows)

expanded_df = expanded_df.sample(frac=1, random_state=42).reset_index(drop=True)

expanded_df.reset_index(drop=True, inplace=True)
expanded_df["id"] = expanded_df.index

expanded_df = expanded_df[["id", "original_row_id", "comment", "rule", "subreddit", "rule_violation", "source_type"]]


In [5]:
print(f"New dataset created with {len(expanded_df)} rows.")
expanded_df.head()


New dataset created with 10145 rows.


,id,original_row_id,comment,rule,subreddit,rule_violation,source_type
0,0,1867,The real zinger here is out of all the stupid ...,No legal advice: Do not offer or request legal...,legaladvice,0,negative_example_1
1,1,1002,Throw yourself off a bridge. Virtually no cost...,No legal advice: Do not offer or request legal...,news,0,negative_example_1
2,2,218,**HD** STREAM 1 [Sky Sports 1 HD STREAM 1](htt...,"No Advertising: Spam, referral links, unsolici...",soccerstreams,0,body
3,3,773,Bring it up without her doing so?\n\nIsn't tha...,No legal advice: Do not offer or request legal...,nosleep,0,negative_example_2
4,4,1838,**Dortmund vs Real Madrid Live Stream**\n\n**<...,"No Advertising: Spam, referral links, unsolici...",worldnews,1,positive_example_2


In [6]:
URL_RE = re.compile(r'((?:https?://|www\.)[^\s<>"\'\]\)]*)', flags=re.IGNORECASE)
SUSPICIOUS_TLDS = {"xyz", "top", "click", "work", "gq", "tk", "cf", "ga", "ml"}
SHORTENERS = {"bit", "tinyurl", "ow", "goo", "t", "is", "rb", "cutt", "rebrandly"}

def _parse_url(u: str):
    u2 = u if u.lower().startswith(("http://", "https://")) else "http://" + u
    p = urlparse(u2); netloc = p.netloc.lower().lstrip("[").rstrip("]")
    if netloc.startswith("www."): netloc = netloc[4:]
    parts = [s for s in netloc.split(".") if s]
    if not parts: return None
    domain_root = parts[-2] if len(parts) >= 2 else parts[0]
    tld = parts[-1] if len(parts) >= 2 else ""
    return p.scheme.lower(), netloc, domain_root, tld, max(0, len(parts)-2), len(u)

def normalize_and_features(text: str):
    if not isinstance(text, str) or not text.strip():
        return text, dict(url_count=0, unique_domain_count=0, has_any_https=0,
                          contains_shortener=0, contains_suspicious_tld=0)

    urls = URL_RE.findall(text)
    parsed = [p for u in urls if (p := _parse_url(u))]
    domains = []; tlds = []; schemes = []
    repl = {}
    for u, (scheme, netloc, root, tld, subd, ulen) in zip(urls, parsed):
        domains.append(root); tlds.append(tld); schemes.append(scheme)
        repl[u] = f"[URL_{root}]"

    def _repl(m): return repl.get(m.group(0), "[URL]")
    normalized = URL_RE.sub(_repl, text)

    feats = dict(
        url_count=len(parsed),
        unique_domain_count=len(set(domains)),
        has_any_https=int(any(s == "https" for s in schemes)),
        contains_shortener=int(any(d in SHORTENERS for d in domains)),
        contains_suspicious_tld=int(any(t in SUSPICIOUS_TLDS for t in tlds)),
    )
    return normalized, feats

def add_url_handling(df: pd.DataFrame):
    norms, feats = [], []
    for txt in df["comment"]:
        n, f = normalize_and_features(txt)
        norms.append(n)
        feats.append(f)
    feat_df = pd.DataFrame(feats)
    df = df.copy()
    df["comment_norm"] = norms
    df = pd.concat([df, feat_df], axis=1)
    return df


In [7]:
norms = []; feats = []
for txt in expanded_df["comment"]:
    n, f = normalize_and_features(txt)
    norms.append(n); feats.append(f)
feat_df = pd.DataFrame(feats)

expanded_df = expanded_df.copy()
expanded_df["comment_norm"] = norms
expanded_df = pd.concat([expanded_df, feat_df], axis=1)

In [8]:
train_df, val_df = train_test_split(
    expanded_df,
    test_size=0.1,             
    stratify=expanded_df["rule_violation"],
    random_state=42
)

In [9]:
train_df.to_csv("DataFolder/expanded_train.csv", index=False)
val_df.to_csv("DataFolder/expanded_val.csv", index=False)


print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")

Train size: 9130
Validation size: 1015


In [11]:
assert "comment" in train_df.columns and "comment_norm" in train_df.columns, "Missing expected text columns!"


In [15]:
sample = train_df.sample(5, random_state=42)[["comment", "comment_norm"]]
for i, row in sample.iterrows():
    print(f"\n=== Example {i} ===")
    print("RAW:\n", row["comment"])
    print("\nNORMALIZED:\n", row["comment_norm"])



=== Example 3436 ===
RAW:
 SD Stream: [ES Link](http://sportsntvlive.com/channel-10/) Mobile Compatibility = No

NORMALIZED:
 SD Stream: [ES Link]([URL_sportsntvlive]) Mobile Compatibility = No

=== Example 1490 ===
RAW:
 Enjoy and please subscribe to https://www.youtube.com/channel/UCKbnNP-mAutX12xv7l7WlbQ For MORE funny fails and other cool content!

NORMALIZED:
 Enjoy and please subscribe to [URL_youtube] For MORE funny fails and other cool content!

=== Example 5939 ===
RAW:
 I actually seriously mutilated someones nipples (really). It was caught on video [here](https://www.youtube.com/watch?v=Va1_ktG5ouU)

NORMALIZED:
 I actually seriously mutilated someones nipples (really). It was caught on video [here]([URL_youtube])

=== Example 7594 ===
RAW:
 roadrunner support number 1-844-659-1035 is in existence for the technical support related to your roadrunner email issue.you can contact us any time on our  road runner support number.we believe in provide genuine support to our client